In [3]:
import os
import pandas as pd
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)



# ── Load data ────────────────────────────────────────────────────────────────
df_results = pd.read_csv("pharmaguard_results.csv")
flagged = df_results[df_results['predicted_fraud'] == 1].head(200).copy()

index = faiss.read_index("pharmaguard_index.faiss")
with open("pharmaguard_chunks.pkl", "rb") as f:
    data = pickle.load(f)
texts   = data['texts']
sources = data['sources']

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# ── RAG retrieval con distance restituita ────────────────────────────────────
def retrieve_regulation(query, top_k=TOP_K):
    """
    Restituisce (chunk_text, source, distance).
    La distanza è il valore L2 del nearest neighbour — più basso = più simile.
    """
    q_emb = embedder.encode([query]).astype('float32')
    distances, indices = index.search(q_emb, top_k)
    idx      = indices[0][0]
    distance = float(distances[0][0])
    return texts[idx][:400], sources[idx], distance


# ── Query builder ─────────────────────────────────────────────────────────────
def build_query(row):
    fraud_type = row['fraud_type']
    drug       = row['generic_name']
    specialty  = row['specialty']
    if fraud_type == 'volume_fraud':
        return f"abnormal high prescription volume reporting requirements {drug}"
    elif fraud_type == 'cost_fraud':
        return f"abnormal reimbursement cost fraud pharmaceutical {drug}"
    elif fraud_type == 'off_label':
        return f"off-label prescription {drug} {specialty} reporting requirements"
    else:
        return f"suspicious prescription pattern {drug}"


# ── Groq explanation ──────────────────────────────────────────────────────────
def generate_explanation(row, regulation, regulation_source):
    prompt = f"""You are a pharmaceutical compliance AI assistant.
Analyze the following suspicious prescription case and generate a clear,
concise explanation for a compliance officer.

CASE DETAILS:
- Drug: {row['generic_name']}
- Prescriber Specialty: {row['specialty']}
- Total Claims: {int(row['total_claims'])}
- Total Cost: ${float(row['total_cost']):,.2f}
- Total Patients: {int(row['total_patients'])}
- Anomaly Score: {float(row['anomaly_score']):.4f}
- Fraud Type Detected: {row['fraud_type']}

RELEVANT REGULATORY REFERENCE ({regulation_source}):
{regulation}

Generate a compliance report with:
1. A one-sentence summary of why this case is suspicious
2. The specific fraud pattern detected
3. The regulatory violation based on the reference above
4. Recommended action for the compliance officer

Keep it concise and professional. Maximum 150 words."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.3
    )
    return response.choices[0].message.content


# ── KILL SWITCH MAIN LOOP ─────────────────────────────────────────────────────
KILL_SWITCH_MESSAGE = (
    "⚠️ ANOMALY MATHEMATICALLY DETECTED — NO REGULATORY MATCH FOUND — "
    "MANUAL REVIEW REQUIRED. "
    "The anomaly detection model flagged this prescriber, but no sufficiently "
    "relevant regulatory reference was found in the knowledge base "
    "(FAISS distance > threshold). "
    "A compliance officer must review this case independently without AI guidance."
)

print(f"Processing {len(flagged)} flagged cases with Kill Switch threshold = {KILL_SWITCH_THRESHOLD}\n")
print("=" * 70)

results = []

for _, row in flagged.iterrows():
    query = build_query(row)
    regulation, reg_source, distance = retrieve_regulation(query)

    # ── KILL SWITCH ───────────────────────────────────────────────────────────
    if distance > KILL_SWITCH_THRESHOLD:
        explanation      = KILL_SWITCH_MESSAGE
        reg_confidence   = "BLOCKED"
        groq_called      = False
    else:
        explanation      = generate_explanation(row, regulation, reg_source)
        reg_confidence   = "OK"
        groq_called      = True
    # ─────────────────────────────────────────────────────────────────────────

    results.append({
        **row.to_dict(),
        'query'            : query,
        'regulation_text'  : regulation,
        'regulation_source': reg_source,
        'faiss_distance'   : round(distance, 4),
        'reg_confidence'   : reg_confidence,
        'groq_called'      : groq_called,
        'explanation'      : explanation,
    })

df_explained = pd.DataFrame(results)

# ── Stats ─────────────────────────────────────────────────────────────────────
total         = len(df_explained)
blocked       = len(df_explained[df_explained['reg_confidence'] == 'BLOCKED'])
explained     = total - blocked

print(f"\nTotal flagged cases   : {total}")
print(f"Explained by Groq     : {explained}  ({explained/total*100:.1f}%)")
print(f"Blocked (Kill Switch) : {blocked}   ({blocked/total*100:.1f}%)")
print(f"\nFAISS distance stats:")
print(df_explained['faiss_distance'].describe().round(4).to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
df_explained.to_csv("pharmaguard_explained.csv", index=False)
print("\nSaved: pharmaguard_explained.csv")
print("Columns:", list(df_explained.columns))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Processing 200 flagged cases with Kill Switch threshold = 1.0



RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqw7vvznfcntn1qfzq9p34z5` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99984, Requested 439. Please try again in 6m5.472s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}